In [5]:
import pandas as pd
import numpy as np

In [ ]:
csv_input = 'test.csv'
csv_output = 'output.csv'

In [7]:
df = pd.read_csv(csv_input, sep=';')

PermissionError: [Errno 13] Permission denied: 'test.xlsx'

In [ ]:
print(df.head(5))

  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0      4        3      4     1     1      3        4   0  11  11  
1      5        3      3     1     1      3        2   9  11  11  
2      4        3      2     2     3      3        6  12  13  12  
3      3        2      2     1     1      5        0  14  14  14  
4      4        3      2     1     2      5        0  11  13  13  

[5 rows x 33 columns]


In [ ]:
df = df.drop(columns=['school', 'address', 'reason', 'activities', 'nursery', 'Dalc', 'Walc'])

In [ ]:
print(df.head(5))

  sex  age famsize Pstatus  Medu  Fedu     Mjob      Fjob guardian  \
0   F   18     GT3       A     4     4  at_home   teacher   mother   
1   F   17     GT3       T     1     1  at_home     other   father   
2   F   15     LE3       T     1     1  at_home     other   mother   
3   F   15     GT3       T     4     2   health  services   mother   
4   F   16     GT3       T     3     3    other     other   father   

   traveltime  ...  internet  romantic famrel freetime goout health absences  \
0           2  ...        no        no      4        3     4      3        4   
1           1  ...       yes        no      5        3     3      3        2   
2           1  ...       yes        no      4        3     2      3        6   
3           1  ...       yes       yes      3        2     2      5        0   
4           1  ...        no        no      4        3     2      5        0   

   G1  G2  G3  
0   0  11  11  
1   9  11  11  
2  12  13  12  
3  14  14  14  
4  11  13  13  

[

In [ ]:
# Map categorical variables to numerical values
df["sex"] = df["sex"].map({"F": 0, "M": 1})
df["famsize"] = df["famsize"].map({"LE3": 0, "GT3": 1})
df["Pstatus"] = df["Pstatus"].map({"T": 1, "A": 0})
df["schoolsup"] = df["schoolsup"].map({"no": 0, "yes": 1})
df["famsup"] = df["famsup"].map({"no": 0, "yes": 1})
df["paid"] = df["paid"].map({"no": 0, "yes": 1})
df["higher"] = df["higher"].map({"no": 0, "yes": 1})
df["internet"] = df["internet"].map({"no": 0, "yes": 1})
df["romantic"] = df["romantic"].map({"no": 0, "yes": 1})

In [ ]:
# one hot encoding for nominal features
df = pd.get_dummies(df, columns=["Mjob", "Fjob"], prefix=["Mjob", "Fjob"])
df = pd.get_dummies(df, columns=["guardian"], prefix="guardian")



In [ ]:
print(df.head(5))

   sex  age  famsize  Pstatus  Medu  Fedu  traveltime  studytime  failures  \
0    0   18        1        0     4     4           2          2         0   
1    0   17        1        1     1     1           1          2         0   
2    0   15        0        1     1     1           1          2         0   
3    0   15        1        1     4     2           1          3         0   
4    0   16        1        1     3     3           1          2         0   

   schoolsup  ...  Mjob_services  Mjob_teacher  Fjob_at_home  Fjob_health  \
0          1  ...          False         False         False        False   
1          0  ...          False         False         False        False   
2          1  ...          False         False         False        False   
3          0  ...          False         False         False        False   
4          0  ...          False         False         False        False   

   Fjob_other  Fjob_services  Fjob_teacher  guardian_father  guardia

In [ ]:
# fill mean value depand on other two in G1 and G2 and G3 if 0, 
grades = ['G1', 'G2', 'G3']
df[grades] = df[grades].replace(0, np.nan)
def fill_grades(row):
    vals = row[grades].values.astype(float)
    for i in range(3):
        if np.isnan(vals[i]):
            others = [vals[j] for j in range(3) if j != i and not np.isnan(vals[j])]
            if others:
                vals[i] = np.floor(np.mean(others))
            else:
                vals[i] = 0  # fallback if all are NaN
    row[grades] = vals
    return row

df = df.apply(fill_grades, axis=1)

In [ ]:
# save cleaned DataFrame
df.to_csv(csv_output, index=False)